# Notebook 5 — Predictive Model: VAR & Granger Causality


A variable X Granger-causes Y if knowing past values of X significantly
improves the prediction of Y beyond what past values of Y alone can tell us.


In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.seasonal import STL
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

OUTPUT_PATH = Path(r"path/to/your/data")

panel = pl.read_parquet(OUTPUT_PATH / "panel_features.parquet")
panel_pd = panel.to_pandas().sort_values(["lhg_country", "date"]).reset_index(drop=True)

HIGH_SIGNAL_COUNTRIES = ["PT", "IT", "ES", "HR", "GR", "FR", "AL", "IE"]
ALL_COUNTRIES = panel_pd["lhg_country"].unique().tolist()

print(f"Panel: {panel_pd.shape}")
print(f"Date range: {panel_pd['date'].min()} -> {panel_pd['date'].max()}")

## Stationarity check

VAR and Granger tests require stationary time series.
We use the Augmented Dickey-Fuller (ADF) test.
Non-stationary series are differenced once.

In [ ]:
def check_stationarity(series: pd.Series, name: str) -> bool:
    """
    ADF test. Returns True if stationary (p < 0.05).
    """
    clean = series.dropna()
    if len(clean) < 20:
        return False
    result = adfuller(clean, autolag='AIC')
    p_value = result[1]
    return p_value < 0.05


def prepare_series(df_country: pd.DataFrame,
                   booking_col: str = "pax",
                   reddit_col: str  = "eng_score_roll7sum") -> pd.DataFrame:
    """
    Deseasonalize bookings, check stationarity, difference if needed.
    Returns a clean 2-column DataFrame ready for VAR/Granger.
    """
    df = df_country.set_index("date").sort_index()

    # Deseasonalize bookings with STL
    pax = df[booking_col].asfreq("D").fillna(0)
    try:
        stl = STL(pax, period=7, robust=True)
        pax_deseas = pax - stl.fit().seasonal
    except Exception:
        pax_deseas = pax

    reddit = df[reddit_col].fillna(0)

    # Check stationarity and difference if needed
    pax_stat    = check_stationarity(pax_deseas, "pax")
    reddit_stat = check_stationarity(reddit, "reddit")

    if not pax_stat:
        pax_deseas = pax_deseas.diff().dropna()
    if not reddit_stat:
        reddit = reddit.diff().dropna()

    # Align both series
    combined = pd.DataFrame({
        "pax":    pax_deseas,
        "reddit": reddit,
    }).dropna()

    return combined


# Test on one country
sample = panel_pd[panel_pd["lhg_country"] == "PT"]
prepared = prepare_series(sample)
print(f"Prepared series shape (PT): {prepared.shape}")
print(prepared.head(5))

## Granger causality test (all countries)

We test at lags 7, 14, 21, 28 days.
A country is flagged as significant if p < 0.05 at any tested lag.

In [ ]:
GRANGER_LAGS   = [7, 14, 21, 28]
REDDIT_FEATURE = "eng_score_roll7sum"   # primary signal from notebook 3
SIGNIFICANCE   = 0.05

granger_results = []

for country in ALL_COUNTRIES:
    df_c = panel_pd[panel_pd["lhg_country"] == country]
    try:
        data = prepare_series(df_c, reddit_col=REDDIT_FEATURE)
        if len(data) < max(GRANGER_LAGS) + 10:
            continue

        # Reddit -> Bookings
        test_fwd = grangercausalitytests(
            data[["pax", "reddit"]], maxlag=max(GRANGER_LAGS), verbose=False
        )
        # Bookings -> Reddit (reverse)
        test_rev = grangercausalitytests(
            data[["reddit", "pax"]], maxlag=max(GRANGER_LAGS), verbose=False
        )

        for lag in GRANGER_LAGS:
            p_fwd = test_fwd[lag][0]["ssr_ftest"][1]
            p_rev = test_rev[lag][0]["ssr_ftest"][1]
            granger_results.append({
                "country":    country,
                "lag":        lag,
                "p_reddit_to_bookings": round(p_fwd, 4),
                "p_bookings_to_reddit": round(p_rev, 4),
                "sig_fwd":    p_fwd < SIGNIFICANCE,
                "sig_rev":    p_rev < SIGNIFICANCE,
            })

    except Exception as e:
        print(f"  Skipped {country}: {e}")
        continue

granger_df = pd.DataFrame(granger_results)

# Summary: which countries show significant Reddit -> Bookings causality
sig_countries = (
    granger_df[granger_df["sig_fwd"]]
    .groupby("country")["lag"]
    .apply(list)
    .reset_index()
    .rename(columns={"lag": "significant_lags"})
)

print(f"Countries with significant Reddit -> Bookings Granger causality (p<0.05):")
print(f"{len(sig_countries)} / {len(ALL_COUNTRIES)} countries\n")
print(sig_countries.to_string(index=False))

In [ ]:
# Granger p-value heatmap
granger_pivot = granger_df.pivot(
    index="country", columns="lag", values="p_reddit_to_bookings"
)
granger_pivot.columns = [f"lag{c}d" for c in granger_pivot.columns]

# Sort by minimum p-value (most significant at top)
granger_pivot = granger_pivot.loc[
    granger_pivot.min(axis=1).sort_values().index
]

fig, ax = plt.subplots(figsize=(9, max(8, len(granger_pivot) * 0.4)))
sns.heatmap(
    granger_pivot,
    annot=True,
    fmt=".3f",
    cmap="RdYlGn_r",   # red = not significant (high p), green = significant (lowp)
    vmin=0,
    vmax=0.2,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "p-value (Reddit -> Bookings)"}
)
ax.axhline(
    sum(granger_pivot.min(axis=1) < SIGNIFICANCE),
    color='black', linewidth=2, linestyle='--'
)
ax.set_title(
    f"Granger causality p-values: Reddit -> Bookings\n"
    f"(green = significant p<{SIGNIFICANCE}, dashed line separates significant countries)",
    fontsize=11
)
ax.set_yticks([])
ax.set_xlabel("Lag")
ax.set_ylabel("Countries")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "granger_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Reverse causality check
# If bookings also Granger-cause Reddit, the relationship is bidirectional
# and harder to interpret as a leading indicator

rev_sig = granger_df[granger_df["sig_rev"]].groupby("country")["lag"].apply(list)
fwd_sig = granger_df[granger_df["sig_fwd"]].groupby("country")["lag"].apply(list)

both_sig   = set(rev_sig.index) & set(fwd_sig.index)
reddit_only = set(fwd_sig.index) - set(rev_sig.index)

print("Directionality analysis")
print(f"Reddit -> Bookings only (clean leading indicator) : {sorted(reddit_only)}")
print(f"Bidirectional (both directions significant)       : {sorted(both_sig)}")
print()
print("Countries with Reddit-only causality are the strongest candidates")
print("for the relevance score - Reddit leads bookings without feedback loop.")

## VAR model on high-signal countries

We fit a bivariate VAR on (deseasonalized bookings, Reddit engagement)
for each high-signal country and evaluate forecast accuracy
against a naive baseline.

In [ ]:
FORECAST_HORIZON = 14    # days ahead to forecast
TEST_SIZE        = 60    # hold out last 60 days for evaluation

var_results = []

for country in HIGH_SIGNAL_COUNTRIES:
    df_c = panel_pd[panel_pd["lhg_country"] == country]
    try:
        data = prepare_series(df_c, reddit_col=REDDIT_FEATURE)

        if len(data) < TEST_SIZE + FORECAST_HORIZON + 30:
            print(f"  {country}: insufficient data, skipping")
            continue

        # Train/test split
        train = data.iloc[:-TEST_SIZE]
        test  = data.iloc[-TEST_SIZE:]

        # Fit VAR — select lag order automatically
        model    = VAR(train)
        lag_order = model.select_order(maxlags=14)
        best_lag  = lag_order.selected_orders.get('aic', 7)
        best_lag  = max(1, min(best_lag, 14))  # bound between 1 and 14

        var_fit  = model.fit(best_lag)

        # Rolling forecast over test period
        actuals    = []
        forecasts  = []
        naive_preds = []

        for i in range(0, TEST_SIZE - FORECAST_HORIZON, FORECAST_HORIZON):
            history   = pd.concat([train, test.iloc[:i]])
            var_refit = VAR(history).fit(best_lag)
            forecast  = var_refit.forecast(
                history.values[-best_lag:], steps=FORECAST_HORIZON
            )
            pred_pax  = forecast[:, 0]   # first column = pax
            actual    = test.iloc[i:i+FORECAST_HORIZON]["pax"].values
            naive     = np.repeat(history["pax"].iloc[-1], FORECAST_HORIZON)

            forecasts.append(pred_pax)
            actuals.append(actual)
            naive_preds.append(naive)

        if not actuals:
            continue

        actuals    = np.concatenate(actuals)
        forecasts  = np.concatenate(forecasts)
        naive_preds = np.concatenate(naive_preds)

        mae_var   = mean_absolute_error(actuals, forecasts)
        mae_naive = mean_absolute_error(actuals, naive_preds)
        rmse_var  = np.sqrt(mean_squared_error(actuals, forecasts))
        rmse_naive = np.sqrt(mean_squared_error(actuals, naive_preds))

        # Skill score: how much better than naive? (positive = better)
        mae_skill = (mae_naive - mae_var) / mae_naive * 100

        var_results.append({
            "country":    country,
            "lag_order":  best_lag,
            "mae_var":    round(mae_var, 2),
            "mae_naive":  round(mae_naive, 2),
            "rmse_var":   round(rmse_var, 2),
            "rmse_naive": round(rmse_naive, 2),
            "mae_skill":  round(mae_skill, 1),   # % improvement over naive
        })
        print(f"  {country}: lag={best_lag}, MAE skill={mae_skill:.1f}%")

    except Exception as e:
        print(f"  {country} failed: {e}")
        continue

var_df = pd.DataFrame(var_results)
print("\nVAR forecast results")
print(var_df.to_string(index=False))

In [ ]:
# VAR vs Naive comparison plot
if len(var_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # MAE comparison
    x = np.arange(len(var_df))
    width = 0.35
    axes[0].bar(x - width/2, var_df["mae_var"],   width, label="VAR",   color="steelblue")
    axes[0].bar(x + width/2, var_df["mae_naive"], width, label="Naive", color="lightgray")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(var_df["country"])
    axes[0].set_title(f"MAE: VAR vs Naive ({FORECAST_HORIZON}d forecast)")
    axes[0].set_ylabel("MAE (deseasonalized pax)")
    axes[0].legend()

    # Skill score
    colors = ["steelblue" if s > 0 else "crimson" for s in var_df["mae_skill"]]
    axes[1].bar(var_df["country"], var_df["mae_skill"], color=colors)
    axes[1].axhline(0, color='black', linewidth=0.8)
    axes[1].set_title("MAE skill score (% improvement over naive)")
    axes[1].set_ylabel("% improvement")
    axes[1].set_xlabel("Country")

    for i, (_, row) in enumerate(var_df.iterrows()):
        axes[1].text(i, row["mae_skill"] + 0.5,
                     f"{row['mae_skill']:.1f}%",
                     ha='center', va='bottom', fontsize=9)

    plt.suptitle("VAR model forecast performance — high-signal countries", fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_PATH / "var_forecast_performance.png", dpi=150)
    plt.show()

In [ ]:
# Visual forecast for best country
if len(var_df) > 0:
    best_country = var_df.loc[var_df["mae_skill"].idxmax(), "country"]
    print(f"Plotting forecast for best country: {best_country}")

    df_c  = panel_pd[panel_pd["lhg_country"] == best_country]
    data  = prepare_series(df_c, reddit_col=REDDIT_FEATURE)
    train = data.iloc[:-TEST_SIZE]
    test  = data.iloc[-TEST_SIZE:]

    best_lag = int(var_df.loc[var_df["country"] == best_country, "lag_order"].values[0])
    var_fit  = VAR(train).fit(best_lag)
    forecast = var_fit.forecast(data.values[-best_lag:], steps=FORECAST_HORIZON)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(train.index[-60:], train["pax"].iloc[-60:],
            color="steelblue", linewidth=1.5, label="Training data")
    ax.plot(test.index, test["pax"],
            color="steelblue", linewidth=1.5, linestyle="--", label="Actual (test)")
    ax.plot(
        pd.date_range(test.index[-1], periods=FORECAST_HORIZON+1, freq='D')[1:],
        forecast[:, 0],
        color="crimson", linewidth=2, label=f"VAR forecast ({FORECAST_HORIZON}d)"
    )
    ax.axvline(test.index[0], color='gray', linestyle=':', linewidth=1)
    ax.set_title(f"{best_country} — VAR forecast (deseasonalized bookings)")
    ax.set_ylabel("Deseasonalized pax")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT_PATH / f"var_forecast_{best_country}.png", dpi=150)
    plt.show()

## Save Outputs

In [ ]:
granger_df.to_csv(OUTPUT_PATH / "granger_results.csv", index=False)
sig_countries.to_csv(OUTPUT_PATH / "granger_significant_countries.csv", index=False)

if len(var_df) > 0:
    var_df.to_csv(OUTPUT_PATH / "var_forecast_results.csv", index=False)

print("Saved:")
print("  outputs/granger_results.csv")
print("  outputs/granger_significant_countries.csv")
print("  outputs/var_forecast_results.csv")